**Building the First Pixel-wise Classifier**

Step 1: Imports and Data Preparation

In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm

Step 2: Create the Custom Dataset Class
This section is where we modify the dataset to work with image patches (pieces of the image).

In [6]:
class MNISTPatchDataset(Dataset):
    def __init__(self, dataset, patch_size=7, stride=4):
        """
        dataset: MNIST or any other dataset
        patch_size: Size of each patch (patch_size x patch_size)
        stride: How far the sliding window moves between patches
        """
        self.dataset = dataset
        self.patch_size = patch_size
        self.stride = stride
        self.patches, self.labels = self._generate_patches()

    def _generate_patches(self):
        patches = []
        labels = []

        for img, label in self.dataset:
            # Convert image to numpy array (C, H, W) format
            img = np.array(img)

            # Since MNIST is grayscale, the image shape will be (1, H, W),
            # we need to access only the H and W dimensions
            H, W = img.shape[1], img.shape[2]  # Ignore the channel dimension (1)

            # Slide a window over the image to extract patches
            for i in range(0, H - self.patch_size, self.stride):
                for j in range(0, W - self.patch_size, self.stride):
                    patch = img[0, i:i+self.patch_size, j:j+self.patch_size]  # Extract the patch
                    patches.append(patch)
                    labels.append(label)

        return patches, labels

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        patch = torch.tensor(self.patches[idx]).float() / 255.0  # Normalize pixel values
        label = self.labels[idx]
        return patch, label


Step 3: Define the Model

In [7]:
class PatchClassifier(nn.Module):
    def __init__(self):
        super(PatchClassifier, self).__init__()
        self.fc1 = nn.Linear(7*7, 128)  # Fully connected layer 1
        self.fc2 = nn.Linear(128, 10)   # Fully connected layer 2 (10 classes for MNIST digits)

    def forward(self, x):
        x = x.view(-1, 7*7)  # Flatten the 7x7 patches into a vector of size 49
        x = torch.relu(self.fc1(x))  # Apply the first fully connected layer with ReLU activation
        x = self.fc2(x)  # Apply the second fully connected layer to get 10 output values
        return x


Step 4: Training Loop

In [8]:
def train(model, train_loader, optimizer, criterion):
    model.train()  # Set the model to training mode
    pbar = tqdm(train_loader, desc="Training Epoch")  # Show a progress bar
    for xb, yb in pbar:  # Loop through batches of data
        optimizer.zero_grad()  # Zero out previous gradients
        output = model(xb)  # Forward pass through the model
        loss = criterion(output, yb)  # Calculate loss
        loss.backward()  # Backpropagate gradients
        optimizer.step()  # Update model weights
        pbar.set_postfix(loss=loss.item())  # Display the current loss value


Step 5: Data Loading and Model Initialization

In [9]:
# Load MNIST dataset and prepare patches
transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="mnist", train=True, transform=transform, download=True)

patch_size = 7
stride = 4
train_patch_ds = MNISTPatchDataset(train_ds, patch_size=patch_size, stride=stride)

train_loader = DataLoader(train_patch_ds, batch_size=64, shuffle=True)

# Initialize model, optimizer, and loss function
model = PatchClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


Step 6: Train the Model

In [ ]:
# Training the model for 5 epochs
for epoch in range(5):  # Train for 5 epochs
    print(f"Epoch {epoch + 1}")
    train(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch + 1} completed.")
